In [ ]:
import pandas as pd
import numpy as np
import h3
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV, cross_validate
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [ ]:
RANDOM_SEED = 42

In [ ]:
def metricas(y, pred):
    return {"R2": r2_score(y, pred),
            "MAE": mean_absolute_error(y, pred),
            "RMSE": float(np.sqrt(mean_squared_error(y, pred)))}

In [ ]:
def buscar(train_val, target, features, param_dist, cv, n_iter=30):
    rf = RandomForestRegressor(bootstrap=True, n_jobs=-1, random_state=RANDOM_SEED)
    search = RandomizedSearchCV(
        rf, param_dist, n_iter=n_iter, cv=cv,
        scoring="neg_root_mean_squared_error",
        random_state=RANDOM_SEED, n_jobs=1, verbose=0,
    )
    search.fit(train_val[features], train_val[target])
    return search

In [ ]:
def get_folds(df, temps, col_temp, n_folds=3, tam_val=6, gap=0):
    folds = []
    for k in range(n_folds, 0, -1):
        fim_val = len(temps) - tam_val * (k - 1)
        ini_val = fim_val - tam_val
        idx_tr  = np.where(df[col_temp].isin(temps[: ini_val - gap]))[0]
        idx_val = np.where(df[col_temp].isin(temps[ini_val:fim_val]))[0]
        folds.append((idx_tr, idx_val))
    return folds

In [ ]:
def metricas_cv(search, train_val, target, features, cv):
    res = cross_validate(
        clone(search.best_estimator_), train_val[features], train_val[target], cv=cv,
        scoring={"R2": "r2", "MAE": "neg_mean_absolute_error", "RMSE": "neg_root_mean_squared_error"},
    )
    return pd.DataFrame({"R2": res["test_R2"], "MAE": -res["test_MAE"], "RMSE": -res["test_RMSE"]},
                        index=[f"fold {i}" for i in range(1, len(cv) + 1)])